# 04 — LazyPredict Lider Modeliyle Application Domain ve SHAP

Bu v5 sürümünde bütün kod hücreleri eğitim amacıyla satır satır açıklanmıştır. Her çalıştırılabilir satırın hemen üzerinde, o satırın görevini özetleyen kısa bir `#` notu bulunur.

Bu notebook yalnızca **uygulama alanı ve yorumlanabilirlik** analizi yapar.

- Notebook 03 tarafından kaydedilen LazyPredict lider modeli, imputer ve bundle kullanılır.
- Model tekrar eğitilmez.
- LazyPredict lideri yerine başka bir ağaç modeli kurulmaz.
- Feature filtreleme tekrar edilmez.
- PCA kullanılmaz.
- Applicability domain, doğrudan standardize edilmiş Mordred uzayında Ledoit–Wolf Mahalanobis uzaklığıyla hesaplanır.
- Uzaklık eşiği eğitim setinin `%95` yüzdelik değeridir.
- SHAP yöntemi lider model ailesine göre otomatik seçilir.
- Ağaç ve doğrusal modellerde hızlı açıklayıcılar, diğer modellerde model-bağımsız permutation SHAP kullanılır.


## 1 — Paketler

Bu bölüm, notebookun ihtiyaç duyduğu Python paketlerini kontrol eder. Eksik paketler yalnızca gerektiğinde kurulur; böylece aynı kurulum komutları gereksiz yere tekrar çalıştırılmaz. Hücre tamamlandığında sonraki bölümlerde kullanılacak temel yazılımlar hazır olur.

In [ ]:
# importlib.util paketini kullanılabilir hâle getirir.
import importlib.util
# subprocess paketini kullanılabilir hâle getirir.
import subprocess
# sys paketini kullanılabilir hâle getirir.
import sys

# `PACKAGES` değişkenine bu adımda kullanılacak değeri atar.
PACKAGES = [
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("numpy", "numpy"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("pandas", "pandas"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("requests", "requests"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("sklearn", "scikit-learn"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("joblib", "joblib"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("matplotlib", "matplotlib"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("shap", "shap"),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# Belirtilen koleksiyondaki öğeleri sırayla işler.
for import_name, pip_name in PACKAGES:
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if importlib.util.find_spec(import_name) is None:
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        subprocess.check_call(
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            [sys.executable, "-m", "pip", "install", "-q", pip_name]
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Paket kontrolü tamamlandı.")


## 2 — Kütüphaneler ve ayarlar

Bu bölümde kullanılacak kütüphaneler belleğe alınır ve pipeline boyunca değişmeden kullanılacak temel ayarlar tanımlanır. Target ID, klasör yolları, dosya adları ve tekrar üretilebilirlik değerleri burada merkezi olarak tutulur.

Notebook 04 artık sabit bir final ağaç modeli adı kullanmaz. Gerçek lider model dosyasının adı Notebook 03 tarafından oluşturulan bundle içinden okunur. Model-bağımsız SHAP gerektiğinde hesaplama süresini kontrol etmek için ayrı örneklem ve background sınırları tanımlanır.


In [ ]:
# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from pathlib import Path
# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from io import BytesIO
# Gerekli Python paketini kullanılabilir hâle getirir.
import json

# Gerekli Python paketini kullanılabilir hâle getirir.
import joblib
# Gerekli Python paketini kullanılabilir hâle getirir.
import matplotlib.pyplot as plt
# Gerekli Python paketini kullanılabilir hâle getirir.
import numpy as np
# Gerekli Python paketini kullanılabilir hâle getirir.
import pandas as pd
# Gerekli Python paketini kullanılabilir hâle getirir.
import requests
# Gerekli Python paketini kullanılabilir hâle getirir.
import shap

# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from google.colab import drive
# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from IPython.display import display

# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from sklearn.covariance import LedoitWolf
# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from sklearn.preprocessing import StandardScaler

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
drive.mount("/content/drive", force_remount=False)

# `DRIVE_DIR` değişkenine bu adımda kullanılacak değeri atar.
DRIVE_DIR = Path(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "/content/drive/MyDrive/MOL_FET_regression_pipeline"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# `CONFIG_PATH` değişkenine bu adımda kullanılacak değeri atar.
CONFIG_PATH = DRIVE_DIR / "pipeline_config.json"

# `DEFAULT_TARGET_ID` değişkenine bu adımda kullanılacak değeri atar.
DEFAULT_TARGET_ID = "CHEMBL206"
# `DEFAULT_GITHUB_RAW_BASE` değişkenine bu adımda kullanılacak değeri atar.
DEFAULT_GITHUB_RAW_BASE = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "https://raw.githubusercontent.com/"
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "MOL-FEST/MOL-FET_regression/main"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if CONFIG_PATH.exists():
    # `config` değişkenine bu adımda kullanılacak değeri atar.
    config = json.loads(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        CONFIG_PATH.read_text(
            # `encoding` değişkenine bu adımda kullanılacak değeri atar.
            encoding="utf-8"
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
# Önceki koşullar sağlanmadığında alternatif kod bloğunu çalıştırır.
else:
    # `config` değişkenine bu adımda kullanılacak değeri atar.
    config = {
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "target_id": DEFAULT_TARGET_ID,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "github_raw_base": DEFAULT_GITHUB_RAW_BASE,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "feature_filename": f"{DEFAULT_TARGET_ID}_Mordred2D_features.csv",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "imputer_filename": f"{DEFAULT_TARGET_ID}_feature_imputer.joblib",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "bundle_filename": f"{DEFAULT_TARGET_ID}_model_bundle.json",
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    }

# `TARGET_ID` değişkenine bu adımda kullanılacak değeri atar.
TARGET_ID = config["target_id"]
# `GITHUB_RAW_BASE` değişkenine bu adımda kullanılacak değeri atar.
GITHUB_RAW_BASE = config["github_raw_base"]
# `FEATURE_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
FEATURE_FILENAME = config["feature_filename"]
# `IMPUTER_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
IMPUTER_FILENAME = config["imputer_filename"]
# `BUNDLE_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
BUNDLE_FILENAME = config["bundle_filename"]

# `DEFAULT_LEADER_MODEL_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
DEFAULT_LEADER_MODEL_FILENAME = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    f"{TARGET_ID}_LazyPredict_leader_model.joblib"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `MAX_SHAP_SAMPLES` değişkenine bu adımda kullanılacak değeri atar.
MAX_SHAP_SAMPLES = 200
# `MAX_MODEL_AGNOSTIC_SHAP_SAMPLES` değişkenine bu adımda kullanılacak değeri atar.
MAX_MODEL_AGNOSTIC_SHAP_SAMPLES = 30
# `MAX_SHAP_BACKGROUND` değişkenine bu adımda kullanılacak değeri atar.
MAX_SHAP_BACKGROUND = 50
# `DISTANCE_PERCENTILE` değişkenine bu adımda kullanılacak değeri atar.
DISTANCE_PERCENTILE = 95

# Gerekli klasörü mevcut değilse oluşturur.
DRIVE_DIR.mkdir(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    parents=True,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    exist_ok=True,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

## 3 — Drive öncelikli lider model ve pipeline dosyaları

Bu bölüm feature store, model bundle, imputer ve lider model nesnesini yükler.

Önce bundle okunur. Böylece lider modelin gerçek dosya adı ve LazyPredict sıralamasındaki model adı öğrenilir. Daha sonra aynı lider model önce Google Drive'dan, Drive'da bulunamazsa GitHub `artifacts/` klasöründen yüklenir.

Notebook bu aşamada model eğitmez.


In [ ]:
# `load_csv` adlı yardımcı fonksiyonu tanımlar.
def load_csv(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    drive_filename,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    github_relative_path,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
):
    # `drive_path` değişkenine bu adımda kullanılacak değeri atar.
    drive_path = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        DRIVE_DIR / drive_filename
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if drive_path.exists():
        # Fonksiyon sonucunu çağrıldığı yere döndürür.
        return pd.read_csv(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            drive_path,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            low_memory=False,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

    # `url` değişkenine bu adımda kullanılacak değeri atar.
    url = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{GITHUB_RAW_BASE}/"
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{github_relative_path}"
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `response` değişkenine bu adımda kullanılacak değeri atar.
    response = requests.get(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        url,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        timeout=120,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    response.raise_for_status()

    # Fonksiyon sonucunu çağrıldığı yere döndürür.
    return pd.read_csv(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        BytesIO(response.content),
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        low_memory=False,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )


# `load_json` adlı yardımcı fonksiyonu tanımlar.
def load_json(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    drive_filename,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    github_relative_path,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
):
    # `drive_path` değişkenine bu adımda kullanılacak değeri atar.
    drive_path = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        DRIVE_DIR / drive_filename
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if drive_path.exists():
        # Fonksiyon sonucunu çağrıldığı yere döndürür.
        return json.loads(
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            drive_path.read_text(
                # `encoding` değişkenine bu adımda kullanılacak değeri atar.
                encoding="utf-8"
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            )
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

    # `url` değişkenine bu adımda kullanılacak değeri atar.
    url = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{GITHUB_RAW_BASE}/"
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{github_relative_path}"
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `response` değişkenine bu adımda kullanılacak değeri atar.
    response = requests.get(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        url,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        timeout=120,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    response.raise_for_status()

    # Fonksiyon sonucunu çağrıldığı yere döndürür.
    return response.json()


# `load_joblib` adlı yardımcı fonksiyonu tanımlar.
def load_joblib(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    drive_filename,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    github_relative_path,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
):
    # `drive_path` değişkenine bu adımda kullanılacak değeri atar.
    drive_path = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        DRIVE_DIR / drive_filename
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if drive_path.exists():
        # Fonksiyon sonucunu çağrıldığı yere döndürür.
        return joblib.load(
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            drive_path
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

    # `url` değişkenine bu adımda kullanılacak değeri atar.
    url = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{GITHUB_RAW_BASE}/"
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{github_relative_path}"
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `response` değişkenine bu adımda kullanılacak değeri atar.
    response = requests.get(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        url,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        timeout=120,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    response.raise_for_status()

    # `temporary_path` değişkenine bu adımda kullanılacak değeri atar.
    temporary_path = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        Path("/content")
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        / drive_filename
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    temporary_path.write_bytes(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        response.content
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Fonksiyon sonucunu çağrıldığı yere döndürür.
    return joblib.load(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        temporary_path
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )


# `feature_df` değişkenine bu adımda kullanılacak değeri atar.
feature_df = load_csv(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    FEATURE_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    f"data/{FEATURE_FILENAME}",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `bundle` değişkenine bu adımda kullanılacak değeri atar.
bundle = load_json(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    BUNDLE_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    f"artifacts/{BUNDLE_FILENAME}",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `leader_model_filename` değişkenine bu adımda kullanılacak değeri atar.
leader_model_filename = bundle.get(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "leader_model_filename",
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    bundle.get(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "model_filename",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        DEFAULT_LEADER_MODEL_FILENAME,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `leader_model_name` değişkenine bu adımda kullanılacak değeri atar.
leader_model_name = bundle.get(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "leader_model_name",
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    bundle.get(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "selected_model_name",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "UnknownLazyPredictLeader",
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `final_model` değişkenine bu adımda kullanılacak değeri atar.
final_model = load_joblib(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    leader_model_filename,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    f"artifacts/{leader_model_filename}",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `imputer` değişkenine bu adımda kullanılacak değeri atar.
imputer = load_joblib(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    IMPUTER_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    f"artifacts/{IMPUTER_FILENAME}",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Kontrol veya sonuç bilgisini ekrana yazdırır.
print(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "LazyPredict lider modeli:",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    leader_model_name,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Kontrol veya sonuç bilgisini ekrana yazdırır.
print(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Lider model dosyası:",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    leader_model_filename,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

## 4 — Kayıtlı feature sırası ve split

Bu bölüm molekülleri sayısal 2D Mordred descriptorlarına dönüştürür. Yüksek eksik oranlı, bütün moleküllerde aynı olan ve mutlak korelasyonu 0.80'i aşan gereksiz featurelar çıkarılır. Eksik kalan hücrelerin imputasyonu burada değil, eğitim verisi ayrıldıktan sonra Notebook 03'te yapılır.

In [ ]:
# `feature_names` değişkenine bu adımda kullanılacak değeri atar.
feature_names = bundle["feature_names"]
# `metadata_columns` değişkenine bu adımda kullanılacak değeri atar.
metadata_columns = bundle["metadata_columns"]
# `target_column` değişkenine bu adımda kullanılacak değeri atar.
target_column = bundle["target_column"]

# `train_indices` değişkenine bu adımda kullanılacak değeri atar.
train_indices = np.asarray(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    bundle["train_indices"],
    # `dtype` değişkenine bu adımda kullanılacak değeri atar.
    dtype=int,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# `test_indices` değişkenine bu adımda kullanılacak değeri atar.
test_indices = np.asarray(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    bundle["test_indices"],
    # `dtype` değişkenine bu adımda kullanılacak değeri atar.
    dtype=int,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `missing_features` değişkenine bu adımda kullanılacak değeri atar.
missing_features = set(feature_names).difference(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    feature_df.columns
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if missing_features:
    # Koşul sağlanmadığında açıklayıcı bir hata oluşturarak işlemi durdurur.
    raise KeyError(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"Feature store içinde eksik featurelar: "
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{sorted(missing_features)[:10]}"
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# `X_raw` değişkenine bu adımda kullanılacak değeri atar.
X_raw = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    feature_df[feature_names]
    # Belirtilen fonksiyonu tablonun ilgili satır veya sütunlarına uygular.
    .apply(pd.to_numeric, errors="coerce")
    # Belirtilen değerleri güvenli alternatif değerlerle değiştirir.
    .replace([np.inf, -np.inf], np.nan)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `y` değişkenine bu adımda kullanılacak değeri atar.
y = pd.to_numeric(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    feature_df[target_column],
    # `errors` değişkenine bu adımda kullanılacak değeri atar.
    errors="coerce",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `X_train` değişkenine bu adımda kullanılacak değeri atar.
X_train = pd.DataFrame(
    # Daha önce öğrenilmiş dönüşümü yeni veriye uygular.
    imputer.transform(X_raw.iloc[train_indices]),
    # `columns` değişkenine bu adımda kullanılacak değeri atar.
    columns=feature_names,
    # `index` değişkenine bu adımda kullanılacak değeri atar.
    index=train_indices,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `X_test` değişkenine bu adımda kullanılacak değeri atar.
X_test = pd.DataFrame(
    # Daha önce öğrenilmiş dönüşümü yeni veriye uygular.
    imputer.transform(X_raw.iloc[test_indices]),
    # `columns` değişkenine bu adımda kullanılacak değeri atar.
    columns=feature_names,
    # `index` değişkenine bu adımda kullanılacak değeri atar.
    index=test_indices,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `y_train` değişkenine bu adımda kullanılacak değeri atar.
y_train = y.iloc[train_indices]
# `y_test` değişkenine bu adımda kullanılacak değeri atar.
y_test = y.iloc[test_indices]

# Eğitilmiş modelle hedef değer tahminleri üretir.
train_prediction = final_model.predict(X_train)
# Eğitilmiş modelle hedef değer tahminleri üretir.
test_prediction = final_model.predict(X_test)

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Train:", X_train.shape)
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Test:", X_test.shape)


## 5 — Standardize edilmiş Mordred uzayı

Bu bölüm molekülleri sayısal 2D Mordred descriptorlarına dönüştürür. Yüksek eksik oranlı, bütün moleküllerde aynı olan ve mutlak korelasyonu 0.80'i aşan gereksiz featurelar çıkarılır. Eksik kalan hücrelerin imputasyonu burada değil, eğitim verisi ayrıldıktan sonra Notebook 03'te yapılır.

In [ ]:
# `scaler` değişkenine bu adımda kullanılacak değeri atar.
scaler = StandardScaler()

# Dönüştürücüyü bu veri üzerinde öğrenir ve aynı veriyi dönüştürür.
X_train_scaled = scaler.fit_transform(X_train)
# Daha önce öğrenilmiş dönüşümü yeni veriye uygular.
X_test_scaled = scaler.transform(X_test)

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("PCA kullanılmadı.")
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("AD uzayı:", X_train_scaled.shape)


## 6 — Mahalanobis applicability domain

Bu bölüm modelin güvenilir kimyasal uzayını değerlendirir. PCA kullanılmadan, standardize edilmiş Mordred uzayında Ledoit–Wolf Mahalanobis uzaklığı hesaplanır. Eğitim uzaklıklarının yüzde 95 eşiği ve standartlaştırılmış artık sınırı birlikte kullanılarak AD-in ve AD-out kayıtları belirlenir.

In [ ]:
# Modeli veya dönüştürücüyü verilen eğitim verisi üzerinde öğrenir.
distance_model = LedoitWolf().fit(X_train_scaled)

# `train_distance` değişkenine bu adımda kullanılacak değeri atar.
train_distance = np.sqrt(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    distance_model.mahalanobis(X_train_scaled)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# `test_distance` değişkenine bu adımda kullanılacak değeri atar.
test_distance = np.sqrt(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    distance_model.mahalanobis(X_test_scaled)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `distance_threshold` değişkenine bu adımda kullanılacak değeri atar.
distance_threshold = float(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    np.percentile(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        train_distance,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        DISTANCE_PERCENTILE,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `train_residual` değişkenine bu adımda kullanılacak değeri atar.
train_residual = y_train.to_numpy() - train_prediction
# `test_residual` değişkenine bu adımda kullanılacak değeri atar.
test_residual = y_test.to_numpy() - test_prediction

# `residual_scale` değişkenine bu adımda kullanılacak değeri atar.
residual_scale = float(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    np.std(train_residual, ddof=1)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if not np.isfinite(residual_scale) or residual_scale < 1e-8:
    # `residual_scale` değişkenine bu adımda kullanılacak değeri atar.
    residual_scale = 1.0

# `train_standardized_residual` değişkenine bu adımda kullanılacak değeri atar.
train_standardized_residual = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    train_residual / residual_scale
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# `test_standardized_residual` değişkenine bu adımda kullanılacak değeri atar.
test_standardized_residual = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    test_residual / residual_scale
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `train_ad_in` değişkenine bu adımda kullanılacak değeri atar.
train_ad_in = (
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    (train_distance <= distance_threshold)
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    & (np.abs(train_standardized_residual) <= 3)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `test_ad_in` değişkenine bu adımda kullanılacak değeri atar.
test_ad_in = (
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    (test_distance <= distance_threshold)
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    & (np.abs(test_standardized_residual) <= 3)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Uzaklık eşiği:", distance_threshold)
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Train AD-in:", int(train_ad_in.sum()), "/", len(train_ad_in))
# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Test AD-in:", int(test_ad_in.sum()), "/", len(test_ad_in))


## 7 — AD tabloları ve grafik

Bu bölüm model sonuçlarını sayısal metrikler veya grafiklerle değerlendirir. Train, test ve çapraz doğrulama sonuçları ayrı tutulur; böylece aşırı uyum, genellenebilirlik ve hata büyüklüğü birlikte yorumlanabilir.

In [ ]:
# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
train_ad = feature_df.iloc[train_indices][metadata_columns].copy()
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
train_ad["actual"] = y_train.to_numpy()
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
train_ad["predicted"] = train_prediction
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
train_ad["mahalanobis_distance"] = train_distance
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
train_ad["standardized_residual"] = train_standardized_residual
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
train_ad["AD_status"] = np.where(train_ad_in, "AD-in", "AD-out")

# Orijinal veriyi değiştirmemek için bağımsız bir kopya oluşturur.
test_ad = feature_df.iloc[test_indices][metadata_columns].copy()
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
test_ad["actual"] = y_test.to_numpy()
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
test_ad["predicted"] = test_prediction
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
test_ad["mahalanobis_distance"] = test_distance
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
test_ad["standardized_residual"] = test_standardized_residual
# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
test_ad["AD_status"] = np.where(test_ad_in, "AD-in", "AD-out")

# `train_ad_path` değişkenine bu adımda kullanılacak değeri atar.
train_ad_path = DRIVE_DIR / f"{TARGET_ID}_train_AD.csv"
# `test_ad_path` değişkenine bu adımda kullanılacak değeri atar.
test_ad_path = DRIVE_DIR / f"{TARGET_ID}_test_AD.csv"
# `ad_plot_path` değişkenine bu adımda kullanılacak değeri atar.
ad_plot_path = DRIVE_DIR / f"{TARGET_ID}_AD_distance_residual.png"

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
train_ad.to_csv(train_ad_path, index=False)
# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
test_ad.to_csv(test_ad_path, index=False)

# Yeni bir matplotlib grafik alanı oluşturur.
plt.figure(figsize=(9, 6))
# İki değişken arasındaki ilişkiyi saçılım grafiğiyle gösterir.
plt.scatter(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    train_distance,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    train_standardized_residual,
    # `alpha` değişkenine bu adımda kullanılacak değeri atar.
    alpha=0.5,
    # `label` değişkenine bu adımda kullanılacak değeri atar.
    label="Train",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# İki değişken arasındaki ilişkiyi saçılım grafiğiyle gösterir.
plt.scatter(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    test_distance,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    test_standardized_residual,
    # `alpha` değişkenine bu adımda kullanılacak değeri atar.
    alpha=0.8,
    # `marker` değişkenine bu adımda kullanılacak değeri atar.
    marker="^",
    # `label` değişkenine bu adımda kullanılacak değeri atar.
    label="Test",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# Grafiğe dikey referans çizgisi ekler.
plt.axvline(distance_threshold, linestyle="--")
# Grafiğe yatay referans çizgisi ekler.
plt.axhline(3, linestyle="--")
# Grafiğe yatay referans çizgisi ekler.
plt.axhline(-3, linestyle="--")
# Grafiğin yatay eksen etiketini tanımlar.
plt.xlabel("Mahalanobis distance")
# Grafiğin dikey eksen etiketini tanımlar.
plt.ylabel("Standardized residual")
# Grafiğe açıklayıcı bir başlık ekler.
plt.title(f"{TARGET_ID} — Applicability Domain")
# Grafikteki veri gruplarının açıklama kutusunu gösterir.
plt.legend()
# Grafik elemanlarının üst üste gelmesini azaltır.
plt.tight_layout()
# Oluşturulan grafiği belirtilen dosyaya kaydeder.
plt.savefig(ad_plot_path, dpi=300, bbox_inches="tight")
# Hazırlanan grafiği notebook ekranında gösterir.
plt.show()


## 8 — Lider model için uyarlanabilir SHAP değerleri

Bu bölüm Notebook 03'te seçilip Drive'a kaydedilen aynı LazyPredict lider modelini açıklar.

SHAP yöntemi lider modelin yapısına göre seçilir:

1. Lider model bir pipeline ise son estimator ve önceki preprocessing adımları ayrılır.
2. Ağaç tabanlı estimatorlarda `TreeExplainer` kullanılır.
3. Katsayı içeren doğrusal estimatorlarda `LinearExplainer` kullanılır.
4. Diğer model ailelerinde lider modelin `predict` fonksiyonu üzerinde model-bağımsız `PermutationExplainer` kullanılır.

Model-bağımsız SHAP daha maliyetli olduğu için test örneklemi 30, background örneklemi 50 kayıtla sınırlandırılır. Model bu hücrede yeniden eğitilmez.


In [ ]:
# `transformed_feature_names` adlı yardımcı fonksiyonu tanımlar.
def transformed_feature_names(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    preprocessing_pipeline,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    transformed_width,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
):
    # Hata oluşturabilecek işlemi güvenli biçimde dener.
    try:
        # `names` değişkenine bu adımda kullanılacak değeri atar.
        names = (
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            preprocessing_pipeline
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            .get_feature_names_out()
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            .tolist()
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )
    # Belirtilen hata oluşursa kontrollü alternatif işlemi çalıştırır.
    except Exception:
        # `names` değişkenine bu adımda kullanılacak değeri atar.
        names = []

    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if len(names) != transformed_width:
        # `names` değişkenine bu adımda kullanılacak değeri atar.
        names = [
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            f"transformed_feature_{index}"
            # Koleksiyondaki öğeleri sırayla işler.
            for index in range(
                # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                transformed_width
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            )
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ]

    # Fonksiyon sonucunu çağrıldığı yere döndürür.
    return names


# `to_dense_array` adlı yardımcı fonksiyonu tanımlar.
def to_dense_array(matrix):
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if hasattr(matrix, "toarray"):
        # Fonksiyon sonucunu çağrıldığı yere döndürür.
        return matrix.toarray()

    # Fonksiyon sonucunu çağrıldığı yere döndürür.
    return np.asarray(matrix)


# `pipeline_steps` değişkenine bu adımda kullanılacak değeri atar.
pipeline_steps = getattr(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    final_model,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "steps",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    None,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `uses_pipeline_preprocessing` değişkenine bu adımda kullanılacak değeri atar.
uses_pipeline_preprocessing = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    isinstance(pipeline_steps, list)
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    and len(pipeline_steps) >= 2
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if uses_pipeline_preprocessing:
    # `preprocessing_pipeline` değişkenine bu adımda kullanılacak değeri atar.
    preprocessing_pipeline = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        final_model[:-1]
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `shap_estimator` değişkenine bu adımda kullanılacak değeri atar.
    shap_estimator = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        final_model.steps[-1][1]
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `estimator_name` değişkenine bu adımda kullanılacak değeri atar.
    estimator_name = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        shap_estimator
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        .__class__
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        .__name__
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
# Önceki koşullar sağlanmadığında alternatif kod bloğunu çalıştırır.
else:
    # `preprocessing_pipeline` değişkenine bu adımda kullanılacak değeri atar.
    preprocessing_pipeline = None
    # `shap_estimator` değişkenine bu adımda kullanılacak değeri atar.
    shap_estimator = final_model
    # `estimator_name` değişkenine bu adımda kullanılacak değeri atar.
    estimator_name = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        final_model
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        .__class__
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        .__name__
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# `tree_name_tokens` değişkenine bu adımda kullanılacak değeri atar.
tree_name_tokens = (
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Tree",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Forest",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Boost",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "XGB",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "LGBM",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "CatBoost",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `is_tree_estimator` değişkenine bu adımda kullanılacak değeri atar.
is_tree_estimator = any(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    token in estimator_name
    # Koleksiyondaki öğeleri sırayla işler.
    for token in tree_name_tokens
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `is_linear_estimator` değişkenine bu adımda kullanılacak değeri atar.
is_linear_estimator = hasattr(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    shap_estimator,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "coef_",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    uses_pipeline_preprocessing
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    and (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        is_tree_estimator
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        or is_linear_estimator
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
):
    # `sample_size` değişkenine bu adımda kullanılacak değeri atar.
    sample_size = min(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        MAX_SHAP_SAMPLES,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        len(X_test),
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `X_shap_original` değişkenine bu adımda kullanılacak değeri atar.
    X_shap_original = X_test.sample(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        n=sample_size,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        random_state=42,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `background_size` değişkenine bu adımda kullanılacak değeri atar.
    background_size = min(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        MAX_SHAP_BACKGROUND,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        len(X_train),
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `X_background_original` değişkenine bu adımda kullanılacak değeri atar.
    X_background_original = X_train.sample(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        n=background_size,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        random_state=42,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `X_shap_array` değişkenine bu adımda kullanılacak değeri atar.
    X_shap_array = to_dense_array(
        # Daha önce öğrenilen dönüşümü yeni veriye uygular.
        preprocessing_pipeline.transform(
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            X_shap_original
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `X_background_array` değişkenine bu adımda kullanılacak değeri atar.
    X_background_array = to_dense_array(
        # Daha önce öğrenilen dönüşümü yeni veriye uygular.
        preprocessing_pipeline.transform(
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            X_background_original
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `shap_feature_names` değişkenine bu adımda kullanılacak değeri atar.
    shap_feature_names = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        transformed_feature_names(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            preprocessing_pipeline,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            X_shap_array.shape[1],
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `X_shap_explain` değişkenine bu adımda kullanılacak değeri atar.
    X_shap_explain = pd.DataFrame(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        X_shap_array,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        index=X_shap_original.index,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        columns=shap_feature_names,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `X_background_explain` değişkenine bu adımda kullanılacak değeri atar.
    X_background_explain = pd.DataFrame(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        X_background_array,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        index=X_background_original.index,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        columns=shap_feature_names,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if is_tree_estimator:
        # Ağaç tabanlı lider model için hızlı SHAP açıklayıcısı oluşturur.
        explainer = shap.TreeExplainer(
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            shap_estimator
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

        # `SHAP_METHOD` değişkenine bu adımda kullanılacak değeri atar.
        SHAP_METHOD = "TreeExplainer"
    # Önceki koşullar sağlanmadığında alternatif kod bloğunu çalıştırır.
    else:
        # Doğrusal lider model için SHAP açıklayıcısı oluşturur.
        explainer = shap.LinearExplainer(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            shap_estimator,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            X_background_explain,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

        # `SHAP_METHOD` değişkenine bu adımda kullanılacak değeri atar.
        SHAP_METHOD = "LinearExplainer"


    # `explain_rows` adlı yardımcı fonksiyonu tanımlar.
    def explain_rows(dataframe):
        # `transformed_array` değişkenine bu adımda kullanılacak değeri atar.
        transformed_array = to_dense_array(
            # Daha önce öğrenilen dönüşümü yeni veriye uygular.
            preprocessing_pipeline.transform(
                # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                dataframe
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            )
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

        # `transformed_frame` değişkenine bu adımda kullanılacak değeri atar.
        transformed_frame = pd.DataFrame(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            transformed_array,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            index=dataframe.index,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            columns=shap_feature_names,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

        # Fonksiyon sonucunu çağrıldığı yere döndürür.
        return explainer(
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            transformed_frame
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )


# Önceki koşullar sağlanmadığında alternatif kod bloğunu çalıştırır.
else:
    # `sample_size` değişkenine bu adımda kullanılacak değeri atar.
    sample_size = min(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        MAX_MODEL_AGNOSTIC_SHAP_SAMPLES,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        len(X_test),
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `X_shap_original` değişkenine bu adımda kullanılacak değeri atar.
    X_shap_original = X_test.sample(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        n=sample_size,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        random_state=42,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `background_size` değişkenine bu adımda kullanılacak değeri atar.
    background_size = min(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        MAX_SHAP_BACKGROUND,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        len(X_train),
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `X_background_explain` değişkenine bu adımda kullanılacak değeri atar.
    X_background_explain = X_train.sample(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        n=background_size,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        random_state=42,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `shap_feature_names` değişkenine bu adımda kullanılacak değeri atar.
    shap_feature_names = feature_names

    # Her model ailesiyle çalışabilen model-bağımsız SHAP açıklayıcısı oluşturur.
    explainer = shap.Explainer(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        final_model.predict,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        X_background_explain,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        feature_names=shap_feature_names,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        algorithm="permutation",
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `SHAP_METHOD` değişkenine bu adımda kullanılacak değeri atar.
    SHAP_METHOD = "PermutationExplainer"


    # `explain_rows` adlı yardımcı fonksiyonu tanımlar.
    def explain_rows(dataframe):
        # `minimum_evaluations` değişkenine bu adımda kullanılacak değeri atar.
        minimum_evaluations = (
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            2 * len(shap_feature_names)
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            + 1
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

        # Fonksiyon sonucunu çağrıldığı yere döndürür.
        return explainer(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            dataframe,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            max_evals=minimum_evaluations,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )


# `shap_values` değişkenine bu adımda kullanılacak değeri atar.
shap_values = explain_rows(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    X_shap_original
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if shap_values.values.ndim == 3:
    # `shap_values` değişkenine bu adımda kullanılacak değeri atar.
    shap_values = shap_values[
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        :,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        :,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        0,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ]

# `shap_importance` değişkenine bu adımda kullanılacak değeri atar.
shap_importance = pd.DataFrame(
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    {
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "feature": shap_feature_names,
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "mean_abs_SHAP": np.abs(
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            shap_values.values
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ).mean(axis=0),
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    }
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
).sort_values(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "mean_abs_SHAP",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    ascending=False,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
).reset_index(
    # `drop` değişkenine bu adımda kullanılacak değeri atar.
    drop=True
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `importance_path` değişkenine bu adımda kullanılacak değeri atar.
importance_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    / f"{TARGET_ID}_SHAP_importance.csv"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
shap_importance.to_csv(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    importance_path,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    index=False,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    float_format="%.6f",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Kontrol veya sonuç bilgisini ekrana yazdırır.
print(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "SHAP yöntemi:",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    SHAP_METHOD,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Tabloyu notebook ekranında okunabilir biçimde gösterir.
display(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    shap_importance.head(20)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

## 9 — Lider model SHAP grafikleri

Bu bölüm bir önceki hücrede seçilen SHAP yöntemiyle lider modelin grafiklerini üretir.

Global beeswarm ve bar grafikleri featureların genel önemini gösterir. Waterfall grafiği test setindeki en yüksek mutlak hataya sahip molekülü açıklar. Dependence grafiği en önemli featureın değeri ile tahmine katkısı arasındaki ilişkiyi gösterir.

Bütün grafikler aynı LazyPredict lider modelinden elde edilir.


In [ ]:
# `beeswarm_path` değişkenine bu adımda kullanılacak değeri atar.
beeswarm_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    / f"{TARGET_ID}_SHAP_beeswarm.png"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `bar_path` değişkenine bu adımda kullanılacak değeri atar.
bar_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    / f"{TARGET_ID}_SHAP_bar.png"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `waterfall_path` değişkenine bu adımda kullanılacak değeri atar.
waterfall_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    / f"{TARGET_ID}_SHAP_waterfall.png"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `dependence_path` değişkenine bu adımda kullanılacak değeri atar.
dependence_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    / f"{TARGET_ID}_SHAP_dependence.png"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# SHAP sonuçlarını seçilen grafik türüyle görselleştirir.
shap.plots.beeswarm(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    shap_values,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    max_display=20,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    show=False,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Grafik elemanlarının üst üste gelmesini azaltır.
plt.tight_layout()

# Grafiği yüksek çözünürlüklü dosya olarak kaydeder.
plt.savefig(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    beeswarm_path,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    dpi=300,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    bbox_inches="tight",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan grafiği notebook ekranında gösterir.
plt.show()

# SHAP sonuçlarını seçilen grafik türüyle görselleştirir.
shap.plots.bar(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    shap_values,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    max_display=20,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    show=False,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Grafik elemanlarının üst üste gelmesini azaltır.
plt.tight_layout()

# Grafiği yüksek çözünürlüklü dosya olarak kaydeder.
plt.savefig(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    bar_path,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    dpi=300,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    bbox_inches="tight",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan grafiği notebook ekranında gösterir.
plt.show()

# `local_position` değişkenine bu adımda kullanılacak değeri atar.
local_position = int(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    np.argmax(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        np.abs(
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            y_test.to_numpy()
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            - test_prediction
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `X_local_original` değişkenine bu adımda kullanılacak değeri atar.
X_local_original = X_test.iloc[
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    [
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        local_position
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ]
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# `local_values` değişkenine bu adımda kullanılacak değeri atar.
local_values = explain_rows(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    X_local_original
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if local_values.values.ndim == 3:
    # `local_values` değişkenine bu adımda kullanılacak değeri atar.
    local_values = local_values[
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        :,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        :,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        0,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ]

# SHAP sonuçlarını seçilen grafik türüyle görselleştirir.
shap.plots.waterfall(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    local_values[0],
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    max_display=15,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    show=False,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Grafik elemanlarının üst üste gelmesini azaltır.
plt.tight_layout()

# Grafiği yüksek çözünürlüklü dosya olarak kaydeder.
plt.savefig(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    waterfall_path,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    dpi=300,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    bbox_inches="tight",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan grafiği notebook ekranında gösterir.
plt.show()

# `top_feature` değişkenine bu adımda kullanılacak değeri atar.
top_feature = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    shap_importance
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    .iloc[0]["feature"]
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# SHAP sonuçlarını seçilen grafik türüyle görselleştirir.
shap.plots.scatter(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    shap_values[
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        :,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        top_feature,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ],
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    show=False,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Grafik elemanlarının üst üste gelmesini azaltır.
plt.tight_layout()

# Grafiği yüksek çözünürlüklü dosya olarak kaydeder.
plt.savefig(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    dependence_path,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    dpi=300,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    bbox_inches="tight",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan grafiği notebook ekranında gösterir.
plt.show()

## 10 — Analiz özeti

Bu bölüm üretilen veri, model veya analiz sonuçlarını kalıcı dosyalara kaydeder. Dosya adları pipeline config ile uyumlu tutulur; böylece sonraki notebook aynı çıktıyı yeniden hesaplamadan doğrudan kullanabilir.

In [ ]:
# `summary` değişkenine bu adımda kullanılacak değeri atar.
summary = {
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "target_id": TARGET_ID,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "leader_model_name": leader_model_name,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "leader_model_filename": leader_model_filename,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "model_name": final_model.__class__.__name__,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "n_features": len(feature_names),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "shap_feature_count": len(shap_feature_names),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "shap_method": SHAP_METHOD,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "ad_method": "Ledoit-Wolf Mahalanobis distance",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pca_used": False,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "distance_percentile": DISTANCE_PERCENTILE,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "distance_threshold": distance_threshold,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "train_AD_in": int(train_ad_in.sum()),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "train_AD_out": int((~train_ad_in).sum()),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "test_AD_in": int(test_ad_in.sum()),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "test_AD_out": int((~test_ad_in).sum()),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "top_SHAP_feature": str(top_feature),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
}

# `summary_path` değişkenine bu adımda kullanılacak değeri atar.
summary_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    / f"{TARGET_ID}_AD_SHAP_summary.json"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Metin veya JSON içeriğini belirtilen dosyaya kaydeder.
summary_path.write_text(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    json.dumps(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        summary,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        ensure_ascii=False,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        indent=2,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    encoding="utf-8",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Tabloyu notebook ekranında okunabilir biçimde gösterir.
display(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    pd.DataFrame(
        # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
        {
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            "metric": list(
                # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                summary.keys()
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            ),
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            "value": list(
                # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                summary.values()
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            ),
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        }
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)